Notebook com a versao original do codigo publicado no artigo

1. Traditional Gradient Boosting Regressor


In [2]:
# Required Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



In [5]:
# Load Dataset (replace 'file_path' with your file location in Colab)
file_path = './content/dados_jamovi.xlsx' # Upload your dataset to Colab
data = pd.read_excel(file_path)
# Target and Feature Selection
target_column = 'Velocity (mean) [µm/s]'
features_columns = [col for col in data.columns if col != target_column]
# Prepare Data
X = data[features_columns]
y = data[target_column]
# Impute Missing Values
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
# Define the Gradient Boosting Regressor
traditional_gbr_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
random_state=42)


In [6]:

# Perform 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_predictions = cross_val_predict(traditional_gbr_model, X_imputed, y, cv=kf)
# Calculate Metrics for Cross-Validation
cv_mae = mean_absolute_error(y, cv_predictions)
cv_rmse = np.sqrt(mean_squared_error(y, cv_predictions))
cv_r2 = r2_score(y, cv_predictions)
# Results
print("5-Fold Cross-Validation Metrics:")
print(f"MAE: {cv_mae}")
print(f"RMSE: {cv_rmse}")
print(f"R^2: {cv_r2}")


5-Fold Cross-Validation Metrics:
MAE: 234.31456825890695
RMSE: 397.48962627853695
R^2: 0.8868770918785022


2. Physics-Informed Machine Learning Gradient Boosting Regressor


In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
# from google.colab import files
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [9]:
# Upload Manual do Arquivo
# uploaded = files.upload()
# file_name = list(uploaded.keys())[0]
file_name = file_path


In [11]:

# Physical Constants for Feature Engineeringg 
g = 9.81 # Gravitational acceleration (m/s2)
rho_p = 2600 # Particle density (kg/m3)
rho_f = 1000 # Fluid density (kg/m3)
mu = 0.001 # Dynamic viscosity of water (Pa·s)
# Carregar o Arquivo
data = pd.read_excel(file_name)
# Target and Feature Selection
target_column = 'Velocity (mean) [µm/s]'
features_columns = [col for col in data.columns if col != target_column]
# Prepare Data
X = data[features_columns]
y = data[target_column]
# Physics-Informed Feature Engineering
data['Radius (m)'] = data['Radius (max) (mean) [µm]'] * 1e-6 # Convert μm to meters
data['Area (m2)'] = data['Area (mean) [µm²]'] * 1e-12 # Convert μm2 to m2
# Correct Reynolds Number
data['Re'] = (rho_f * data[target_column] * 1e-6 * data['Radius (m)']) / mu # Re = (rho * v * r) /
mu
# Adjust Drag Coefficient for Low Reynolds Number (Stokes regime)
data['Cd'] = 24 / data['Re'] # Cd = 24/Re for laminar regime
# Correct Drag Force using the adjusted Cd
data['Drag Force'] = 0.5 * rho_f * data['Cd'] * data['Area (m2)'] * ((data[target_column] * 1e-6)
** 2) # Fd = 0.5 * rho * Cd * A * v^2
# Stokes' Settling Velocity (para referência)
data['Stokes Velocity'] = (2 / 9) * ((rho_p - rho_f) * g * (data['Radius (m)'] ** 2) / mu)
# Include Physics-Informed Features
physics_features = ['Radius (m)', 'Re', 'Cd', 'Drag Force']
X_piml = pd.concat([X, data[physics_features]], axis=1)
# Impute Missing Values
imputer = SimpleImputer(strategy='mean')
X_piml_imputed = imputer.fit_transform(X_piml)
# Define the Gradient Boosting Regressor
piml_gbr_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05,
random_state=42)
# Perform 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_predictions = cross_val_predict(piml_gbr_model, X_piml_imputed, y, cv=kf)
# Calculate Metrics for Cross-Validation
cv_mae = mean_absolute_error(y, cv_predictions)
cv_rmse = np.sqrt(mean_squared_error(y, cv_predictions))
cv_r2 = r2_score(y, cv_predictions)
# Results
print("5-Fold Cross-Validation Metrics:")
print(f"MAE: {cv_mae}")
print(f"RMSE: {cv_rmse}")
print(f"R^2: {cv_r2}")

5-Fold Cross-Validation Metrics:
MAE: 136.0491303563949
RMSE: 261.5529512797582
R^2: 0.9510200640158917
